# ACT fine-tune on Colab: balanced X-mark data + image augmentation

Starts from **Low101/act_all_ft50k** (ACT, all 151 episodes, 50k fine-tune)
and fine-tunes on **episodes 51-150** of `Low101/third-act-attempt`
(30 varied-square + 70 X-mark episodes) with LeRobot image augmentation
(colour jitter, sharpness, random affine ±5°, ±5% shift).

Before running: on the laptop run `outputs/portfolio/upload_for_colab.sh`
(uploads the dataset and the ACT checkpoint as private repos).

Runtime: GPU (T4 works, L4/A100 faster). Checkpoints go to Google Drive,
so a disconnect does not lose progress (see the resume cell).

In [ ]:
!nvidia-smi
!python --version   # needs Python >= 3.12

## 1. Install LeRobot (same commit as the laptop)

In [ ]:
!git clone https://github.com/huggingface/lerobot.git /content/lerobot
%cd /content/lerobot
!git checkout 64b23178d5348609c266250d3e1f511eba4c33ff
!apt-get -qq install -y ffmpeg
!pip install -q -e ".[training,dataset]"
!pip uninstall -y -q torchao

## 2. Hugging Face login (token with write access)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Google Drive for checkpoints

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/lerobot_runs"

## 4. Settings
- `STEPS = 25000` with a checkpoint every 10000 steps (saved at 10k,
  20k and the final 25k), so you can test intermediate checkpoints.
- `BATCH = 8` matches the earlier ACT runs (lr 1e-5).

In [ ]:
import os
RUN = "act_xbal_aug_ft25k"
EPISODES = str(list(range(51, 151))).replace(" ", "")   # 100 episodes
STEPS = 25000
SAVE_FREQ = 10000
BATCH = 8
WORKERS = min(4, os.cpu_count())
OUT_DIR = f"{DRIVE}/{RUN}"
print(RUN, OUT_DIR, "workers", WORKERS)

## 5. Train

In [ ]:
!lerobot-train \
  --policy.path=Low101/act_all_ft50k \
  --dataset.repo_id=Low101/third-act-attempt \
  --dataset.episodes="{EPISODES}" \
  --dataset.image_transforms.enable=true \
  --policy.device=cuda \
  --policy.push_to_hub=true \
  --policy.repo_id=Low101/{RUN} \
  --policy.private=true \
  --output_dir={OUT_DIR} \
  --job_name={RUN} \
  --steps={STEPS} \
  --save_freq={SAVE_FREQ} \
  --batch_size={BATCH} \
  --num_workers={WORKERS} \
  --wandb.enable=false

## 5b. Resume after a disconnect
Re-run cells 1-4 first, then this cell (continues from the last
checkpoint saved on Drive).

In [ ]:
!lerobot-train \
  --config_path={OUT_DIR}/checkpoints/last/pretrained_model/train_config.json \
  --resume=true

## 5c. After fine-tuning: make the repo private + upload all checkpoints
Sets `Low101/{RUN}` to private (in case it was created public) and uploads
every checkpoint's `pretrained_model` to `checkpoints/<step>/` in the same
private repo. Optimizer states are not uploaded (they stay on Drive).
For the ablation run, set `RUN = RUN2` and `OUT_DIR = OUT_DIR2` first.

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi

api = HfApi()
repo_id = f"Low101/{RUN}"
api.create_repo(repo_id, repo_type="model", private=True, exist_ok=True)
api.update_repo_settings(repo_id, private=True)

ckpt_root = Path(OUT_DIR) / "checkpoints"
steps = sorted(p.name for p in ckpt_root.iterdir()
               if p.is_dir() and p.name != "last")
for step in steps:
    api.upload_folder(
        repo_id=repo_id,
        folder_path=str(ckpt_root / step / "pretrained_model"),
        path_in_repo=f"checkpoints/{step}",
        commit_message=f"checkpoint {step}",
    )
    print("uploaded", step)

print("private:", api.model_info(repo_id).private)

## 6. (Optional) ablation: augmentation only, all 151 episodes
Tells you how much of the gain comes from augmentation vs. balancing.
Run after section 5 finishes.

In [ ]:
RUN2 = "act_all_aug_ft25k"
OUT_DIR2 = f"{DRIVE}/{RUN2}"
!lerobot-train \
  --policy.path=Low101/act_all_ft50k \
  --dataset.repo_id=Low101/third-act-attempt \
  --dataset.image_transforms.enable=true \
  --policy.device=cuda \
  --policy.push_to_hub=true \
  --policy.repo_id=Low101/{RUN2} \
  --policy.private=true \
  --output_dir={OUT_DIR2} \
  --job_name={RUN2} \
  --steps={STEPS} \
  --save_freq={SAVE_FREQ} \
  --batch_size={BATCH} \
  --num_workers={WORKERS} \
  --wandb.enable=false

## 7. Get the model on the laptop
Final model (pushed automatically at the end of training):
```bash
cd ~/lerobot/lerobot
hf download Low101/act_xbal_aug_ft25k \
  --local-dir outputs/train/act_xbal_aug_ft25k/pretrained_model
```
A specific checkpoint (uploaded by cell 5c), e.g. 10000:
```bash
hf download Low101/act_xbal_aug_ft25k --include "checkpoints/010000/*" \
  --local-dir outputs/train/act_xbal_aug_ft25k
```
Also on Google Drive
`lerobot_runs/act_xbal_aug_ft25k/checkpoints/<step>/pretrained_model`.